# Phishing Email Detection with BERT

## Notebook 01 – Data Preprocessing

This notebook builds the cleaned, combined email dataset used by all downstream training, robustness, and evaluation notebooks.


### Notebook objectives

- Load raw email corpora from `../data`.
- Normalize URLs, punctuation, HTML entities, and quoted-printable text.
- Standardize label values across sources.
- Remove duplicates, empty texts, and invalid labels.
- Shuffle the final dataset and save it as `combined.csv`.


### 1. Imports and paths

Import core libraries and define the base data directory.


In [1]:
import pandas as pd
from pathlib import Path
import glob, re, html, quopri

DATA_DIR = Path("../data")

### 2. Text normalization helpers

Define helper functions to normalize URLs, HTML entities, and email bodies into a consistent text field.


In [2]:
def normalize_punct_spacing_for_urls(s):
    s = str(s)
    s = re.sub(r'\s*\.\s*', '.', s)    # remove spaces around '.'
    s = re.sub(r'\s*/\s*', '/', s)     # remove spaces around '/'
    s = re.sub(r'\s*:\s*', ':', s)     # remove spaces around ':'
    s = re.sub(r'(?i)\b(https?|ftp|file)\s*[\.:;]\s*/\s*/', r'\1://', s)  # https.//, http;// -> https://, http://
    return re.sub(r'\s+', ' ', s).strip()

URL_RE = re.compile(
    r"(?:"
    r"((https?|ftp|file)\s*:[^\s]*)"                                   # http:, https:, ftp:, file:
    r"|"
    r"(www\.[^\s]+)"                                                   # www.example.com
    r"|"
    r"(\b[A-Za-z0-9.-]+\.(com|net|org|info|biz|ru|cn|io|co)\b[^\s]*)"  # bare domain
    r"|"
    r"(\bhttps?\b)"                                                    # bare 'http' or 'https'
    r"|"
    r"(\bwww\s+[A-Za-z0-9.-]+\s+\b(com|net|org|info|biz|ru|cn|io|co)\b)"# 'www anywheremd com'
    r")",
    flags=re.IGNORECASE,
)

HTML_TAG_RE = re.compile(r"<[^>]+>")
HEADER_RE = re.compile(
    r"^(from|to|cc|bcc|subject|date|reply-to|return-path|message-id|received|"
    r"x-[\w-]+|mime-version|content-type|content-transfer-encoding|boundary)\s*:.*$",
    flags=re.IGNORECASE | re.MULTILINE,
)
BOUNDARY_RE = re.compile(r"^--[-_=A-Za-z0-9]+$", flags=re.MULTILINE)

def clean_text_for_bert(text):
    text = str(text)
    
    try:
        text = quopri.decodestring(text).decode("utf-8", "ignore")
    except Exception:
        pass

    text = normalize_punct_spacing_for_urls(text)

    text = HEADER_RE.sub("", text)
    text = BOUNDARY_RE.sub("", text)

    text = HTML_TAG_RE.sub(" ", text)

    text = URL_RE.sub(" [URL] ", text)

    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

### 3. Load and merge raw datasets

Load all individual corpora, clean their labels and text, and concatenate them into a single DataFrame.


In [3]:
dfs = []

files = [
    "CEAS_08.csv",
    "Enron.csv",
    "Ling.csv",
    "Nazario.csv",
    "Nazario_5.csv",
    "SpamAssassin.csv",
    "TREC_05.csv",
    "TREC_06.csv",
    "TREC_07.csv",
    "Nigerian_Fraud.csv",
    "Nigerian_5.csv",
]

READ_KWARGS = {"encoding": "utf-8", "encoding_errors": "ignore", "low_memory": False}

for name in files:
    path = f"../data/{name}"
    print(f"Loading {name} ...")

    df = pd.read_csv(path, **READ_KWARGS)

    df.columns = [c.strip().lower() for c in df.columns]

    if "subject" not in df.columns:
        df["subject"] = ""
    if "body" not in df.columns:
        df["body"] = ""

    df["text"] = (
        df["subject"].astype(str).fillna("") + " " +
        df["body"].astype(str).fillna("")
    ).str.strip()

    df["text"] = df["text"].apply(clean_text_for_bert)

    if "label" not in df.columns:
        print(f"  Skipped {name}: no 'label' column")
        continue

    df = df.dropna(subset=["label"])

    mask_label_allowed = df["label"].isin([0, 1]) | df["label"].astype(str).str.strip().isin(["0", "1"])
    df = df[mask_label_allowed]

    df["label"] = df["label"].astype(int)

    df = df[df["text"] != ""]

    df = df[["text", "label"]]

    print(f"  kept {len(df)} rows")
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)

Loading CEAS_08.csv ...
  kept 39033 rows
Loading Enron.csv ...
  kept 29725 rows
Loading Ling.csv ...
  kept 2859 rows
Loading Nazario.csv ...
  kept 1563 rows
Loading Nazario_5.csv ...
  kept 3046 rows
Loading SpamAssassin.csv ...
  kept 5780 rows
Loading TREC_05.csv ...
  kept 54948 rows
Loading TREC_06.csv ...
  kept 16339 rows
Loading TREC_07.csv ...
  kept 53670 rows
Loading Nigerian_Fraud.csv ...
  kept 3230 rows
Loading Nigerian_5.csv ...
  kept 6206 rows


### 4. Remove duplicate texts

Drop exact duplicate messages based on the `text` column to avoid data leakage across splits.


In [4]:
df_all = pd.concat(dfs, ignore_index=True)

before = len(df_all)
df_all = df_all.drop_duplicates(subset=["text"])
after = len(df_all)
print(f"Removed {before - after} duplicate entries")

Removed 16662 duplicate entries


### 5. Shuffle rows

Shuffle the cleaned dataset with a fixed random seed to ensure reproducible ordering.


In [5]:
df_all = df_all.sample(frac=1, random_state=2025).reset_index(drop=True)
print("After cleaning:", df_all.shape)

After cleaning: (199737, 2)


### 6. Inspect class balance

Check the overall dataset shape and label distribution for ham vs. phishing emails.


In [6]:
print("Shape:", df_all.shape)
print("Label distribution:\n", df_all["label"].value_counts())

df_all.head()

Shape: (199737, 2)
Label distribution:
 label
0    107533
1     92204
Name: count, dtype: int64


,text,label
0,re:uribl > 71.920 77.1604 1.2331 0.984 0.71 0....,0
1,can you last 36 hours ? mol ci - ialis softabs...,1
2,"home delivery cials-tabs ""the beverage.icee, b...",1
3,fw:presentation tammie schoppe enron americas-...,0
4,good day.what's so good about it?:) [url] grab...,1


### 7. Save combined dataset

Write the final preprocessed dataset to `combined.csv` under `../data`.


In [7]:
DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = DATA_DIR / "combined.csv"
df_all.to_csv(output_path, index=False)

print(f"Saved combined dataset to: {output_path.resolve()}")

Saved combined dataset to: /home/nmd/projects/phishing-detector/data/combined.csv
